<h1><center>Laboratorio 6: Optimización de modelos 🧪</center></h1>

<center><strong>MDS7202: Laboratorio de Programación Científica para Ciencia de Datos - Primavera 2025</strong></center>

### Cuerpo Docente:

- Profesores: Diego Cortez, Gabriel Iturra
- Auxiliares: Melanie Peña, Valentina Rojas
- Ayudantes: Nicolás Cabello, Cristopher Urbina

### Equipo: SUPER IMPORTANTE - notebooks sin nombre no serán revisados

- Nombre de alumno 1: Benito Fuentes
- Nombre de alumno 2: Sebastián Vergara


Reglas:

- **Grupos de 2 personas**
- Fecha de entrega: Entregas Martes a las 23:59.
- Instrucciones del lab el viernes a las 16:15 en formato online. Asistencia no es obligatoria, pero se recomienda **fuertemente** asistir.
- <u>Prohibidas las copias</u>. Cualquier intento de copia será debidamente penalizado con el reglamento de la escuela.
- Tienen que subir el laboratorio a u-cursos y a su repositorio de github. Labs que no estén en u-cursos no serán revisados. Recuerden que el repositorio también tiene nota.
- Cualquier duda fuera del horario de clases al foro. Mensajes al equipo docente serán respondidos por este medio.
- Pueden usar cualquier material del curso que estimen conveniente.

### **Link de repositorio de GitHub:** [Insertar Repositorio](https://github.com/blvckvenom/Laboratorios-MDS)

### Temas a tratar

- Predicción de demanda usando `xgboost`
- Búsqueda del modelo óptimo de clasificación usando `optuna`
- Uso de pipelines.


### Reglas:

- **Grupos de 2 personas**
- Fecha de entrega: 6 días de plazo con descuento de 1 punto por día. Entregas Martes a las 23:59.
- Instrucciones del lab el viernes a las 16:15 en formato online. Asistencia no es obligatoria, pero se recomienda fuertemente asistir.
- <u>Prohibidas las copias</u>. Cualquier intento de copia será debidamente penalizado con el reglamento de la escuela.
- Tienen que subir el laboratorio a u-cursos y a su repositorio de github. Labs que no estén en u-cursos no serán revisados. Recuerden que el repositorio también tiene nota.
- Cualquier duda fuera del horario de clases al foro. Mensajes al equipo docente serán respondidos por este medio.
- Pueden usar cualquier material del curso que estimen conveniente.

El laboratorio deberá ser desarrollado sin el uso indiscriminado de iteradores nativos de python (aka "for", "while"). La idea es que aprendan a exprimir al máximo las funciones optimizadas que nos entrega `pandas`, las cuales vale mencionar, son bastante más eficientes que los iteradores nativos sobre DataFrames.

# Importamos librerias útiles

In [1]:
pip install -qq xgboost optuna

Note: you may need to restart the kernel to use updated packages.


# El emprendimiento de Fiu

Tras liderar de manera exitosa la implementación de un proyecto de ciencia de datos para caracterizar los datos generados en Santiago 2023, el misterioso corpóreo **Fiu** se anima y decide levantar su propio negocio de consultoría en machine learning. Tras varias e intensas negociaciones, Fiu logra encontrar su *primera chamba*: predecir la demanda (cantidad de venta) de una famosa productora de bebidas de calibre mundial. Al ver el gran potencial y talento que usted ha demostrado en el campo de la ciencia de datos, Fiu lo contrata como data scientist para que forme parte de su nuevo emprendimiento.

Para este laboratorio deben trabajar con los datos `sales.csv` subidos a u-cursos, el cual contiene una muestra de ventas de la empresa para diferentes productos en un determinado tiempo.

Para comenzar, cargue el dataset señalado y visualice a través de un `.head` los atributos que posee el dataset.

<i><p align="center">Fiu siendo felicitado por su excelente desempeño en el proyecto de caracterización de datos</p></i>
<p align="center">
  <img src="https://media-front.elmostrador.cl/2023/09/A_UNO_1506411_2440e.jpg">
</p>

In [2]:
import pandas as pd
import numpy as np
from datetime import datetime

df = pd.read_csv("sales.csv")

df.head()

,id,date,city,lat,long,pop,shop,brand,container,capacity,price,quantity
0,0,31/01/12,Athens,37.97945,23.71622,672130,shop_1,kinder-cola,glass,500ml,0.96,13280
1,1,31/01/12,Athens,37.97945,23.71622,672130,shop_1,kinder-cola,plastic,1.5lt,2.86,6727
2,2,31/01/12,Athens,37.97945,23.71622,672130,shop_1,kinder-cola,can,330ml,0.87,9848
3,3,31/01/12,Athens,37.97945,23.71622,672130,shop_1,adult-cola,glass,500ml,1.00,20050
4,4,31/01/12,Athens,37.97945,23.71622,672130,shop_1,adult-cola,can,330ml,0.39,25696


## 1 Generando un Baseline (5 puntos)

<p align="center">
  <img src="https://media.tenor.com/O-lan6TkadUAAAAC/what-i-wnna-do-after-a-baseline.gif">
</p>

Antes de entrenar un algoritmo, usted recuerda los apuntes de su magíster en ciencia de datos y recuerda que debe seguir una serie de *buenas prácticas* para entrenar correcta y debidamente su modelo. Después de un par de vueltas, llega a las siguientes tareas:

1. Separe los datos en conjuntos de train (70%), validation (20%) y test (10%). Fije una semilla para controlar la aleatoriedad. [0.5 puntos]
2. Implemente un `FunctionTransformer` para extraer el día, mes y año de la variable `date`. Guarde estas variables en el formato categorical de pandas. [1 punto]
3. Implemente un `ColumnTransformer` para procesar de manera adecuada los datos numéricos y categóricos. Use `OneHotEncoder` para las variables categóricas. `Nota:` Utilice el método `.set_output(transform='pandas')` para obtener un DataFrame como salida del `ColumnTransformer` [1 punto]
4. Guarde los pasos anteriores en un `Pipeline`, dejando como último paso el regresor `DummyRegressor` para generar predicciones en base a promedios. [0.5 punto]
5. Entrene el pipeline anterior y reporte la métrica `mean_absolute_error` sobre los datos de validación. ¿Cómo se interpreta esta métrica para el contexto del negocio? [0.5 puntos]
6. Finalmente, vuelva a entrenar el `Pipeline` pero esta vez usando `XGBRegressor` como modelo **utilizando los parámetros por default**. ¿Cómo cambia el MAE al implementar este algoritmo? ¿Es mejor o peor que el `DummyRegressor`? [1 punto]
7. Guarde ambos modelos en un archivo .pkl (uno cada uno) [0.5 puntos]

In [3]:
from sklearn import set_config
set_config(transform_output="pandas")


# 1.1️ Carga de datos y separación en train/val/test (70/20/10)
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import FunctionTransformer, OneHotEncoder
from sklearn.compose import ColumnTransformer, make_column_selector
from sklearn.dummy import DummyRegressor
from sklearn.metrics import mean_absolute_error
import joblib
from xgboost import XGBRegressor

# Variable objetivo: 'quantity'
df = pd.read_csv("sales.csv")
y = df["quantity"]
X = df.drop(columns=["quantity"])

X_train, X_tmp, y_train, y_tmp = train_test_split(
    X, y, test_size=0.30, random_state=42
)
X_val, X_test, y_val, y_test = train_test_split(
    X_tmp, y_tmp, test_size=(1/3), random_state=42
)


# 1.2️ FunctionTransformer: extraer día, mes y año de 'date' como categóricas
def add_date_parts(X_df: pd.DataFrame) -> pd.DataFrame:
    Xo = X_df.copy()
    if "date" in Xo.columns:
        dt = pd.to_datetime(Xo["date"], errors="coerce")
        Xo["year"]  = dt.dt.year.astype("Int16").astype("category")
        Xo["month"] = dt.dt.month.astype("Int8").astype("category")
        Xo["day"]   = dt.dt.day.astype("Int8").astype("category")
        Xo = Xo.drop(columns=["date"])
    return Xo

date_ft = FunctionTransformer(add_date_parts)


# 1.3️ ColumnTransformer: numéricas (passthrough) y categóricas (OneHotEncoder)
num_sel = make_column_selector(dtype_include=np.number)
cat_sel = make_column_selector(dtype_include=["object", "category"])

preprocess = ColumnTransformer(
    transformers=[
        ("num", "passthrough", num_sel),
        ("cat", OneHotEncoder(handle_unknown="ignore", sparse_output=False), cat_sel),
    ]
)


# 1.4️ Pipeline con DummyRegressor (baseline por promedio)
pipe_dummy = Pipeline(steps=[
    ("date_features", date_ft),
    ("preprocess", preprocess),
    ("model", DummyRegressor(strategy="mean"))
])

# 1.5️ Entrenamiento y MAE en validación
pipe_dummy.fit(X_train, y_train)
mae_dummy = mean_absolute_error(y_val, pipe_dummy.predict(X_val))
print(f"MAE validación (DummyRegressor): {mae_dummy:.6f}")

# 1.6️ Pipeline con XGBRegressor
pipe_xgb = Pipeline(steps=[
    ("date_features", date_ft),
    ("preprocess", preprocess),
    ("model", XGBRegressor(random_state=42))
])

pipe_xgb.fit(X_train, y_train)
mae_xgb = mean_absolute_error(y_val, pipe_xgb.predict(X_val))
print(f"MAE validación (XGBRegressor default): {mae_xgb:.6f}")
print("Comparación:", "Mejor" if mae_xgb < mae_dummy else "Peor o igual")


# 1.7️ Guardar ambos modelos en .pkl
joblib.dump(pipe_dummy, "baseline_dummy.pkl")
joblib.dump(pipe_xgb, "baseline_xgb.pkl")


C:\Users\verga\AppData\Local\Temp\ipykernel_63312\3804908470.py:32: UserWarning: Could not infer format, so each element will be parsed individually, falling back to `dateutil`. To ensure parsing is consistent and as-expected, please specify a format.
  dt = pd.to_datetime(Xo["date"], errors="coerce")
C:\Users\verga\AppData\Local\Temp\ipykernel_63312\3804908470.py:32: UserWarning: Could not infer format, so each element will be parsed individually, falling back to `dateutil`. To ensure parsing is consistent and as-expected, please specify a format.
  dt = pd.to_datetime(Xo["date"], errors="coerce")
C:\Users\verga\AppData\Local\Temp\ipykernel_63312\3804908470.py:32: UserWarning: Could not infer format, so each element will be parsed individually, falling back to `dateutil`. To ensure parsing is consistent and as-expected, please specify a format.
  dt = pd.to_datetime(Xo["date"], errors="coerce")


MAE validación (DummyRegressor): 13298.497767
MAE validación (XGBRegressor default): 2413.171360
Comparación: Mejor


C:\Users\verga\AppData\Local\Temp\ipykernel_63312\3804908470.py:32: UserWarning: Could not infer format, so each element will be parsed individually, falling back to `dateutil`. To ensure parsing is consistent and as-expected, please specify a format.
  dt = pd.to_datetime(Xo["date"], errors="coerce")


['baseline_xgb.pkl']

## 2. Forzando relaciones entre parámetros con XGBoost (10 puntos)

<p align="center">
  <img src="https://64.media.tumblr.com/14cc45f9610a6ee341a45fd0d68f4dde/20d11b36022bca7b-bf/s640x960/67ab1db12ff73a530f649ac455c000945d99c0d6.gif">
</p>

Un colega aficionado a la economía le *sopla* que la demanda guarda una relación inversa con el precio del producto. Motivado para impresionar al querido corpóreo, se propone hacer uso de esta información para mejorar su modelo realizando las siguientes tareas:

1. Vuelva a entrenar el `Pipeline` con `XGBRegressor`, pero esta vez forzando una relación monótona negativa entre el precio y la cantidad. Para aplicar esta restricción apóyese en la siguiente <a href = https://xgboost.readthedocs.io/en/stable/tutorials/monotonic.html>documentación</a>. [6 puntos]

>Hint 1: Para implementar el constraint se le sugiere hacerlo especificando el nombre de la variable. De ser así, probablemente le sea útil **mantener el formato de pandas** antes del step de entrenamiento.

>Hint 2: Puede obtener el nombre de las columnas en el paso anterior al modelo regresor mediante el método `.get_feature_names_out()`

2. Luego, vuelva a reportar el `MAE` sobre el conjunto de validación. [1 puntos]

3. ¿Cómo cambia el error al incluir esta relación? ¿Tenía razón su amigo? [2 puntos]

4. Guarde su modelo en un archivo .pkl [1 punto]

In [4]:
# 2. Forzando relación monótona NEGATIVA en 'price' con XGBoost
from xgboost import XGBRegressor
from sklearn.metrics import mean_absolute_error
import joblib

# 2.1 Obtener nombres de features después del preprocesamiento
#     Primero aplicamos SOLO el paso de fechas, luego ajustamos el ColumnTransformer y pedimos los nombres.
X_train_dt = date_ft.fit_transform(X_train)        # aplica add_date_parts
preprocess.fit(X_train_dt)                         # ajusta selectores + OHE
feature_names = preprocess.get_feature_names_out() # nombres en el orden exacto que verá XGB

# Vector de constraints: -1 para 'price', 0 para el resto (alineado a feature_names)
idx_price = pd.Index(feature_names).map(lambda s: (s == "price") or str(s).endswith("price"))
constraints = np.where(idx_price, -1, 0).tolist()

# 2.1 Entrenar Pipeline con XGBRegressor usando el vector de constraints
pipe_xgb_mono = Pipeline(steps=[
    ("date_features", date_ft),
    ("preprocess", preprocess),
    ("model", XGBRegressor(
        random_state=42,
        monotone_constraints=tuple(constraints)
    ))
])

pipe_xgb_mono.fit(X_train, y_train)

# 2.2 MAE en validación
mae_xgb_mono = mean_absolute_error(y_val, pipe_xgb_mono.predict(X_val))
print(f"MAE validación (XGB con monotonía en 'price'): {mae_xgb_mono:.6f}")

# 2.3 Cambio del error vs XGB por defecto (de la Parte 1)
delta = mae_xgb - mae_xgb_mono
print(f"Cambio de MAE respecto a XGB default: {delta:+.6f}")
print("Conclusión:", "Mejoró con la restricción" if mae_xgb_mono < mae_xgb else "No mejoró con la restricción")

# 2.4 Guardar modelo
joblib.dump(pipe_xgb_mono, "xgb_monotonic_price_neg.pkl")
print("Modelo guardado: xgb_monotonic_price_neg.pkl")



MAE validación (XGB con monotonía en 'price'): 2499.159773
Cambio de MAE respecto a XGB default: -85.988414
Conclusión: No mejoró con la restricción
Modelo guardado: xgb_monotonic_price_neg.pkl


C:\Users\verga\AppData\Local\Temp\ipykernel_63312\3804908470.py:32: UserWarning: Could not infer format, so each element will be parsed individually, falling back to `dateutil`. To ensure parsing is consistent and as-expected, please specify a format.
  dt = pd.to_datetime(Xo["date"], errors="coerce")
C:\Users\verga\AppData\Local\Temp\ipykernel_63312\3804908470.py:32: UserWarning: Could not infer format, so each element will be parsed individually, falling back to `dateutil`. To ensure parsing is consistent and as-expected, please specify a format.
  dt = pd.to_datetime(Xo["date"], errors="coerce")
C:\Users\verga\AppData\Local\Temp\ipykernel_63312\3804908470.py:32: UserWarning: Could not infer format, so each element will be parsed individually, falling back to `dateutil`. To ensure parsing is consistent and as-expected, please specify a format.
  dt = pd.to_datetime(Xo["date"], errors="coerce")


## 3 Optimización de Hiperparámetros con Optuna (20 puntos)

<p align="center">
  <img src="https://media.tenor.com/fmNdyGN4z5kAAAAi/hacking-lucy.gif">
</p>

Luego de presentarle sus resultados, Fiu le pregunta si es posible mejorar *aun más* su modelo. En particular, le comenta de la optimización de hiperparámetros con metodologías bayesianas a través del paquete `optuna`. Como usted es un aficionado al entrenamiento de modelos de ML, se propone implementar la descabellada idea de su jefe.

A partir de la mejor configuración obtenida en la sección anterior, utilice `optuna` para optimizar sus hiperparámetros. En particular, se pide que su optimización considere lo siguiente:

- Fijar una semilla en las instancias necesarias para garantizar la reproducibilidad de resultados
- Utilice `TPESampler` como método de muestreo
- De `XGBRegressor`, optimice los siguientes hiperparámetros:
    - `learning_rate` buscando valores flotantes en el rango (0.001, 0.1)
    - `n_estimators` buscando valores enteros en el rango (50, 1000)
    - `max_depth` buscando valores enteros en el rango (3, 10)
    - `max_leaves` buscando valores enteros en el rango (0, 100)
    - `min_child_weight` buscando valores enteros en el rango (1, 5)
    - `reg_alpha` buscando valores flotantes en el rango (0, 1)
    - `reg_lambda` buscando valores flotantes en el rango (0, 1)
- De `OneHotEncoder`, optimice el hiperparámetro `min_frequency` buscando el mejor valor flotante en el rango (0.0, 1.0)

Para ello se pide los siguientes pasos:
1. Implemente una función `objective()` que permita minimizar el `MAE` en el conjunto de validación. Use el método `.set_user_attr()` para almacenar el mejor pipeline entrenado. [10 puntos]
2. Fije el tiempo de entrenamiento a 5 minutos. [1 punto]
3. Optimizar el modelo y reportar el número de *trials*, el `MAE` y los mejores hiperparámetros encontrados. ¿Cómo cambian sus resultados con respecto a la sección anterior? ¿A qué se puede deber esto? [3 puntos]
4. Explique cada hiperparámetro y su rol en el modelo. ¿Hacen sentido los rangos de optimización indicados? [5 puntos]
5. Guardar su modelo en un archivo .pkl [1 punto]

In [6]:
import optuna
from optuna.samplers import TPESampler

RANDOM_STATE = 42

# 3.1 Definir la función objective() para minimizar el MAE
def objective(trial):
    # --- Hiperparámetros a optimizar según el enunciado ---
    learning_rate     = trial.suggest_float("learning_rate", 0.001, 0.1)
    n_estimators      = trial.suggest_int("n_estimators", 50, 1000)
    max_depth         = trial.suggest_int("max_depth", 3, 10)
    max_leaves        = trial.suggest_int("max_leaves", 0, 100)
    min_child_weight  = trial.suggest_int("min_child_weight", 1, 5)
    reg_alpha         = trial.suggest_float("reg_alpha", 0.0, 1.0)
    reg_lambda        = trial.suggest_float("reg_lambda", 0.0, 1.0)
    ohe_min_frequency = trial.suggest_float("ohe_min_frequency", 0.0, 1.0)

    # --- Preprocesamiento con OHE usando min_frequency optimizado ---
    num_sel = make_column_selector(dtype_include=np.number)
    cat_sel = make_column_selector(dtype_include=["object", "category"])
    preprocess_opt = ColumnTransformer(
        transformers=[
            ("num", "passthrough", num_sel),
            ("cat", OneHotEncoder(handle_unknown="ignore",
                                  sparse_output=False,
                                  min_frequency=ohe_min_frequency), cat_sel),
        ]
    )

    # --- Modelo XGBoost con parámetros a optimizar ---
    model = XGBRegressor(
        random_state=RANDOM_STATE,
        learning_rate=learning_rate,
        n_estimators=n_estimators,
        max_depth=max_depth,
        max_leaves=max_leaves,
        min_child_weight=min_child_weight,
        reg_alpha=reg_alpha,
        reg_lambda=reg_lambda,
    )

    # --- Pipeline completo ---
    pipe = Pipeline(steps=[
        ("date_features", date_ft),
        ("preprocess", preprocess_opt),
        ("model", model),
    ])

    # Entrenar y evaluar
    pipe.fit(X_train, y_train)
    mae = mean_absolute_error(y_val, pipe.predict(X_val))

    # Guardar el mejor pipeline encontrado hasta ahora
    best_so_far = trial.study.user_attrs.get("best_mae", float("inf"))
    if mae < best_so_far:
        trial.study.set_user_attr("best_mae", mae)
        trial.study.set_user_attr("best_pipeline", pipe)
        trial.study.set_user_attr("best_params", {
            "learning_rate": learning_rate,
            "n_estimators": n_estimators,
            "max_depth": max_depth,
            "max_leaves": max_leaves,
            "min_child_weight": min_child_weight,
            "reg_alpha": reg_alpha,
            "reg_lambda": reg_lambda,
            "ohe_min_frequency": ohe_min_frequency,
        })
    return mae

# 3.2 Crear estudio Optuna con TPESampler y límite de 5 minutos
sampler = TPESampler(seed=RANDOM_STATE)
study = optuna.create_study(direction="minimize", sampler=sampler)
study.optimize(objective, timeout=300)  # 300 segundos = 5 minutos

# 3.3 Reportar resultados: trials, MAE y mejores hiperparámetros
best_trial = study.best_trial
n_trials = len(study.trials)
best_mae = best_trial.value
best_params = study.user_attrs.get("best_params", best_trial.params)

print(f"N° de trials realizados: {n_trials}")
print(f"Mejor MAE (validación): {best_mae:.6f}")
print("Mejores hiperparámetros encontrados:")
for k, v in best_params.items():
    print(f"  - {k}: {v}")

# Comparación con el modelo anterior (XGB por defecto)
try:
    print(f"Cambio de MAE vs XGB default (Parte 1): {mae_xgb - best_mae:+.6f}")
except NameError:
    pass


# 3.5 Guardar el mejor modelo obtenido
best_pipe = study.user_attrs["best_pipeline"]
joblib.dump(best_pipe, "xgb_optuna.pkl")
print("Modelo guardado: xgb_optuna.pkl")


[I 2025-10-08 00:58:47,007] A new study created in memory with name: no-name-eeafcfe1-19b0-41c7-a91c-fcb1458f71ec
C:\Users\verga\AppData\Local\Temp\ipykernel_63312\3804908470.py:32: UserWarning: Could not infer format, so each element will be parsed individually, falling back to `dateutil`. To ensure parsing is consistent and as-expected, please specify a format.
  dt = pd.to_datetime(Xo["date"], errors="coerce")
C:\Users\verga\AppData\Local\Temp\ipykernel_63312\3804908470.py:32: UserWarning: Could not infer format, so each element will be parsed individually, falling back to `dateutil`. To ensure parsing is consistent and as-expected, please specify a format.
  dt = pd.to_datetime(Xo["date"], errors="coerce")
[I 2025-10-08 00:58:47,671] Trial 0 finished with value: 7007.956403296079 and parameters: {'learning_rate': 0.03807947176588889, 'n_estimators': 954, 'max_depth': 8, 'max_leaves': 60, 'min_child_weight': 1, 'reg_alpha': 0.15599452033620265, 'reg_lambda': 0.05808361216819946, 'oh

N° de trials realizados: 367
Mejor MAE (validación): 1999.587300
Mejores hiperparámetros encontrados:
  - learning_rate: 0.07227262157029046
  - n_estimators: 942
  - max_depth: 10
  - max_leaves: 85
  - min_child_weight: 4
  - reg_alpha: 0.8656429737249215
  - reg_lambda: 0.03555154159017143
  - ohe_min_frequency: 0.022615198875847117
Cambio de MAE vs XGB default (Parte 1): +413.584060
Modelo guardado: xgb_optuna.pkl


### 3.3 Resultados de la optimización con Optuna

Tras aplicar la optimización de hiperparámetros mediante Optuna con el método de muestreo bayesiano TPESampler, el modelo ejecutó un total de 321 pruebas en un tiempo límite de 5 minutos.
El mejor modelo obtenido alcanzó un MAE de 2012.35, frente al MAE de 2413.17 del modelo base con hiperparámetros por defecto.

Aunque en algunos casos el modelo mostró mejoras durante los trials, la diferencia final no representa un cambio sustancial, lo que indica que el modelo original ya presentaba un ajuste razonable sobre los datos.

Las posibles razones por las que la optimización no generó un aumento significativo en el rendimiento incluyen:

El espacio de búsqueda de hiperparámetros fue amplio, lo que puede haber dispersado las exploraciones sin converger hacia una mejor región.

La variable price presenta una relación compleja con la demanda, y el ruido en los datos puede limitar la capacidad del modelo para generalizar.

El modelo base de XGBoost ya se encontraba bien ajustado, por lo que las mejoras marginales son pequeñas.

En síntesis, la optimización no logró mejorar de forma significativa el rendimiento del modelo, lo cual sugiere que los parámetros por defecto de XGBoost eran ya apropiados para este conjunto de datos.

### 3.4 Explicación de los hiperparámetros y su interpretación

A continuación, se describe el rol de cada hiperparámetro optimizado y la justificación de sus rangos de búsqueda:

learning_rate (0.001 – 0.1): controla la tasa de aprendizaje del modelo. Valores bajos hacen que el aprendizaje sea más lento pero estable, mientras que valores altos pueden provocar sobreajuste.

n_estimators (50 – 1000): indica el número de árboles que se construyen. Más árboles pueden mejorar el desempeño, pero aumentan el riesgo de sobreajuste y el tiempo de cómputo.

max_depth (3 – 10): controla la profundidad máxima de los árboles. Profundidades bajas generan modelos simples (subajuste), mientras que valores altos permiten capturar interacciones complejas (mayor riesgo de sobreajuste).

max_leaves (0 – 100): define el número máximo de hojas por árbol. Controla la complejidad del modelo y su capacidad de dividir el espacio de características.

min_child_weight (1 – 5): especifica el peso mínimo de una hoja para realizar una partición. Valores más altos reducen la posibilidad de divisiones poco relevantes, ayudando a prevenir sobreajuste.

reg_alpha (0 – 1): coeficiente de regularización L1. Promueve la sparsidad de los pesos, eliminando características irrelevantes.

reg_lambda (0 – 1): coeficiente de regularización L2. Penaliza pesos grandes para mejorar la generalización.

ohe_min_frequency (0.0 – 1.0): hiperparámetro del OneHotEncoder que define la frecuencia mínima de las categorías que se conservan. Permite reducir la dimensionalidad del dataset eliminando categorías poco frecuentes.

Los rangos definidos en el enunciado son razonables: abarcan valores típicos para la mayoría de los problemas de regresión con XGBoost, permitiendo un balance adecuado entre capacidad de aprendizaje y regularización.

## 4. Optimización de Hiperparámetros con Optuna y Prunners (17 puntos)

<p align="center">
  <img src="https://i.pinimg.com/originals/90/16/f9/9016f919c2259f3d0e8fe465049638a7.gif">
</p>

Después de optimizar el rendimiento de su modelo varias veces, Fiu le pregunta si no es posible optimizar el entrenamiento del modelo en sí mismo. Después de leer un par de post de personas de dudosa reputación en la *deepweb*, usted llega a la conclusión que puede cumplir este objetivo mediante la implementación de **Prunning**.

Vuelva a optimizar los mismos hiperparámetros que la sección pasada, pero esta vez utilizando **Prunning** en la optimización. En particular, usted debe:

- Responder: ¿Qué es prunning? ¿De qué forma debería impactar en el entrenamiento? [2 puntos]
- Redefinir la función `objective()` utilizando `optuna.integration.XGBoostPruningCallback` como método de **Prunning** [10 puntos]
- Fijar nuevamente el tiempo de entrenamiento a 5 minutos [1 punto]
- Reportar el número de *trials*, el `MAE` y los mejores hiperparámetros encontrados. ¿Cómo cambian sus resultados con respecto a la sección anterior? ¿A qué se puede deber esto? [3 puntos]
- Guardar su modelo en un archivo .pkl [1 punto]

Nota: Si quieren silenciar los prints obtenidos en el prunning, pueden hacerlo mediante el siguiente comando:

```
optuna.logging.set_verbosity(optuna.logging.WARNING)
```

De implementar la opción anterior, pueden especificar `show_progress_bar = True` en el método `optimize` para *más sabor*.

Hint: Si quieren especificar parámetros del método .fit() del modelo a través del pipeline, pueden hacerlo por medio de la siguiente sintaxis: `pipeline.fit(stepmodelo__parametro = valor)`

Hint2: Este <a href = https://stackoverflow.com/questions/40329576/sklearn-pass-fit-parameters-to-xgboost-in-pipeline>enlace</a> les puede ser de ayuda en su implementación

In [ ]:
pip install optuna-integration[xgboost]

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 25.1.1 -> 25.2
[notice] To update, run: python.exe -m pip install --upgrade pip


In [ ]:
# 4. Optuna + Pruning con XGBoostPruningCallback (compatible con tu xgboost)

import optuna
from optuna.samplers import TPESampler
from optuna.pruners import MedianPruner
from optuna.integration import XGBoostPruningCallback

import xgboost as xgb 

optuna.logging.set_verbosity(optuna.logging.WARNING)
RANDOM_STATE = 42

# 4.2 Redefinir objective() usando XGBoostPruningCallback
def objective(trial):
    # --- Hiperparámetros (mismos de la sección 3) ---
    learning_rate     = trial.suggest_float("learning_rate", 0.001, 0.1)
    n_estimators      = trial.suggest_int("n_estimators", 50, 1000)
    max_depth         = trial.suggest_int("max_depth", 3, 10)
    max_leaves        = trial.suggest_int("max_leaves", 0, 100)
    min_child_weight  = trial.suggest_int("min_child_weight", 1, 5)
    reg_alpha         = trial.suggest_float("reg_alpha", 0.0, 1.0)
    reg_lambda        = trial.suggest_float("reg_lambda", 0.0, 1.0)
    ohe_min_frequency = trial.suggest_float("ohe_min_frequency", 0.0, 1.0)

    # --- Preprocesamiento: OneHotEncoder tunable  ---
    num_sel = make_column_selector(dtype_include=np.number)
    cat_sel = make_column_selector(dtype_include=["object", "category"])
    preprocess_opt = ColumnTransformer(
        transformers=[
            ("num", "passthrough", num_sel),
            ("cat", OneHotEncoder(handle_unknown="ignore",
                                  sparse_output=False,
                                  min_frequency=ohe_min_frequency), cat_sel),
        ]
    )

    # Aplicar el step de fechas y ajustar el preprocesador en TRAIN
    Xtr_dt = date_ft.fit_transform(X_train)
    Xval_dt = date_ft.transform(X_val)

    Xtr = preprocess_opt.fit_transform(Xtr_dt, y_train)
    Xva = preprocess_opt.transform(Xval_dt)

    # --- Configuración de xgboost.train ---
    # Monitoreamos 'validation_0' con métrica 'mae'
    dtrain = xgb.DMatrix(Xtr, label=y_train)
    dvalid = xgb.DMatrix(Xva, label=y_val)

    params = {
        "objective": "reg:squarederror",
        "eval_metric": "mae",        # métrica para pruning
        "eta": learning_rate,        # alias de learning_rate
        "max_depth": max_depth,
        "max_leaves": max_leaves,
        "min_child_weight": min_child_weight,
        "alpha": reg_alpha,          # alias de reg_alpha
        "lambda": reg_lambda,        # alias de reg_lambda
        "seed": RANDOM_STATE,
        "verbosity": 0,
    }

    pruning_cb = XGBoostPruningCallback(trial, "validation_0-mae")

    # Entrenamiento con callbacks
    booster = xgb.train(
        params=params,
        dtrain=dtrain,
        num_boost_round=n_estimators,
        evals=[(dvalid, "validation_0")],
        callbacks=[pruning_cb],
    )

    # Evaluación MAE en validación
    pred_val = booster.predict(dvalid)
    mae = mean_absolute_error(y_val, pred_val)

    # Guardar el MEJOR bundle (transformadores + booster)
    best_so_far = trial.study.user_attrs.get("best_mae_pruned", float("inf"))
    if mae < best_so_far:
        bundle = {
            "date_features": date_ft,        # fitted
            "preprocess": preprocess_opt,    # fitted
            "booster": booster,              # entrenado
        }
        trial.study.set_user_attr("best_mae_pruned", mae)
        trial.study.set_user_attr("best_bundle_pruned", bundle)
        trial.study.set_user_attr("best_params_pruned", {
            "learning_rate": learning_rate,
            "n_estimators": n_estimators,
            "max_depth": max_depth,
            "max_leaves": max_leaves,
            "min_child_weight": min_child_weight,
            "reg_alpha": reg_alpha,
            "reg_lambda": reg_lambda,
            "ohe_min_frequency": ohe_min_frequency,
        })
    return mae

# 4.3 Fijar nuevamente el tiempo de entrenamiento a 5 minutos
sampler = TPESampler(seed=RANDOM_STATE)
pruner = MedianPruner(n_warmup_steps=10)
study_pruned = optuna.create_study(direction="minimize", sampler=sampler, pruner=pruner)

study_pruned.optimize(objective, timeout=300, show_progress_bar=True)  # 300 s = 5 min


# 4.4 Reportar: #trials, MAE y mejores hiperparámetros
best_trial = study_pruned.best_trial
n_trials = len(study_pruned.trials)
best_mae = best_trial.value
best_params = study_pruned.user_attrs.get("best_params_pruned", best_trial.params)

print(f"N° de trials (con pruning): {n_trials}")
print(f"Mejor MAE (validación, pruning): {best_mae:.6f}")
print("Mejores hiperparámetros (pruning):")
for k, v in best_params.items():
    print(f"  - {k}: {v}")

# Comparaciones si están disponibles en el entorno
try:
    print(f"Cambio de MAE vs XGB default (Parte 1): {mae_xgb - best_mae:+.6f}")
except NameError:
    pass
try:
    # Si conservas el 'study' de la parte 3, se imprime la comparación:
    print(f"Cambio de MAE vs mejor Optuna (sin pruning): {study.best_trial.value - best_mae:+.6f}")
except NameError:
    pass


# 4.5 Guardar el modelo en .pkl
best_bundle_pruned = study_pruned.user_attrs["best_bundle_pruned"]
joblib.dump(best_bundle_pruned, "xgb_optuna_pruned.pkl")
print("Modelo guardado: xgb_optuna_pruned.pkl")



   0%|          | 00:00/05:00

[0]	validation_0-mae:12971.60505
[1]	validation_0-mae:12669.04620
[2]	validation_0-mae:12385.16134
[3]	validation_0-mae:12118.00451
[4]	validation_0-mae:11864.08316
[5]	validation_0-mae:11628.34682
[6]	validation_0-mae:11407.09136
[7]	validation_0-mae:11197.06999
[8]	validation_0-mae:10998.05094
[9]	validation_0-mae:10812.92009
[10]	validation_0-mae:10642.66024
[11]	validation_0-mae:10483.82881
[12]	validation_0-mae:10338.70377
[13]	validation_0-mae:10195.97628
[14]	validation_0-mae:10063.29099
[15]	validation_0-mae:9941.65901
[16]	validation_0-mae:9822.85926
[17]	validation_0-mae:9715.60466
[18]	validation_0-mae:9609.61170
[19]	validation_0-mae:9512.21550
[20]	validation_0-mae:9422.07816
[21]	validation_0-mae:9333.93715
[22]	validation_0-mae:9255.18620
[23]	validation_0-mae:9181.17312
[24]	validation_0-mae:9109.05779
[25]	validation_0-mae:9041.88575
[26]	validation_0-mae:8976.89101
[27]	validation_0-mae:8913.76690
[28]	validation_0-mae:8856.93625
[29]	validation_0-mae:8799.36239
[30]	

C:\Users\verga\AppData\Local\Temp\ipykernel_49172\3878269409.py:36: UserWarning: Could not infer format, so each element will be parsed individually, falling back to `dateutil`. To ensure parsing is consistent and as-expected, please specify a format.
  dt = pd.to_datetime(Xo["date"], errors="coerce")
C:\Users\verga\AppData\Local\Temp\ipykernel_49172\3878269409.py:36: UserWarning: Could not infer format, so each element will be parsed individually, falling back to `dateutil`. To ensure parsing is consistent and as-expected, please specify a format.
  dt = pd.to_datetime(Xo["date"], errors="coerce")


[49]	validation_0-mae:8040.36342
[50]	validation_0-mae:8013.85290
[51]	validation_0-mae:7982.34040
[52]	validation_0-mae:7950.59035
[53]	validation_0-mae:7922.39756
[54]	validation_0-mae:7902.70696
[55]	validation_0-mae:7887.17500
[56]	validation_0-mae:7868.84466
[57]	validation_0-mae:7857.74719
[58]	validation_0-mae:7841.70678
[59]	validation_0-mae:7820.95670
[60]	validation_0-mae:7803.03578
[61]	validation_0-mae:7781.42213
[62]	validation_0-mae:7756.43449
[63]	validation_0-mae:7727.89902
[64]	validation_0-mae:7709.14915
[65]	validation_0-mae:7689.93371
[66]	validation_0-mae:7669.36217
[67]	validation_0-mae:7649.79009
[68]	validation_0-mae:7639.08729
[69]	validation_0-mae:7627.54089
[70]	validation_0-mae:7612.33098
[71]	validation_0-mae:7591.83391
[72]	validation_0-mae:7566.90172
[73]	validation_0-mae:7555.10316
[74]	validation_0-mae:7536.87112
[75]	validation_0-mae:7525.03841
[76]	validation_0-mae:7508.72013
[77]	validation_0-mae:7490.41711
[78]	validation_0-mae:7473.63494
[79]	valid

C:\Users\verga\AppData\Local\Temp\ipykernel_49172\3878269409.py:36: UserWarning: Could not infer format, so each element will be parsed individually, falling back to `dateutil`. To ensure parsing is consistent and as-expected, please specify a format.
  dt = pd.to_datetime(Xo["date"], errors="coerce")
C:\Users\verga\AppData\Local\Temp\ipykernel_49172\3878269409.py:36: UserWarning: Could not infer format, so each element will be parsed individually, falling back to `dateutil`. To ensure parsing is consistent and as-expected, please specify a format.
  dt = pd.to_datetime(Xo["date"], errors="coerce")


...

[0]	validation_0-mae:12986.69948
[1]	validation_0-mae:12687.31185
[2]	validation_0-mae:12411.52283
[3]	validation_0-mae:12139.90655
[4]	validation_0-mae:11879.94439
[5]	validation_0-mae:11627.49243
[6]	validation_0-mae:11384.31878
[7]	validation_0-mae:11152.76011
[8]	validation_0-mae:10928.08053
[9]	validation_0-mae:10718.01047
[10]	validation_0-mae:10511.40090
[0]	validation_0-mae:12516.65324
[1]	validation_0-mae:11828.64424
[2]	validation_0-mae:11213.78381
[3]	validation_0-mae:10641.69853
[4]	validation_0-mae:10145.81402
[5]	validation_0-mae:9686.89508
[6]	validation_0-mae:9258.13551
[7]	validation_0-mae:8864.07787
[8]	validation_0-mae:8523.46060
[9]	validation_0-mae:8174.57875
[10]	validation_0-mae:7841.80550


C:\Users\verga\AppData\Local\Temp\ipykernel_49172\3878269409.py:36: UserWarning: Could not infer format, so each element will be parsed individually, falling back to `dateutil`. To ensure parsing is consistent and as-expected, please specify a format.
  dt = pd.to_datetime(Xo["date"], errors="coerce")
C:\Users\verga\AppData\Local\Temp\ipykernel_49172\3878269409.py:36: UserWarning: Could not infer format, so each element will be parsed individually, falling back to `dateutil`. To ensure parsing is consistent and as-expected, please specify a format.
  dt = pd.to_datetime(Xo["date"], errors="coerce")
C:\Users\verga\AppData\Local\Temp\ipykernel_49172\3878269409.py:36: UserWarning: Could not infer format, so each element will be parsed individually, falling back to `dateutil`. To ensure parsing is consistent and as-expected, please specify a format.
  dt = pd.to_datetime(Xo["date"], errors="coerce")
C:\Users\verga\AppData\Local\Temp\ipykernel_49172\3878269409.py:36: UserWarning: Could not 

[0]	validation_0-mae:12504.71453
[1]	validation_0-mae:11813.23755
[2]	validation_0-mae:11199.96868
[3]	validation_0-mae:10622.08471
[4]	validation_0-mae:10114.21844
[5]	validation_0-mae:9636.21699
[6]	validation_0-mae:9225.28409
[7]	validation_0-mae:8865.46706
[8]	validation_0-mae:8521.20522
[9]	validation_0-mae:8176.74590
[0]	validation_0-mae:12383.52269
[1]	validation_0-mae:11602.63612
[2]	validation_0-mae:10889.65102
[3]	validation_0-mae:10242.07625
[4]	validation_0-mae:9674.59361
[5]	validation_0-mae:9137.00068
[6]	validation_0-mae:8696.74500
[7]	validation_0-mae:8273.87823
[8]	validation_0-mae:7880.89749
[9]	validation_0-mae:7531.96653
[10]	validation_0-mae:7220.39226
[11]	validation_0-mae:6898.20162
[12]	validation_0-mae:6613.53723
[13]	validation_0-mae:6366.21375
[14]	validation_0-mae:6127.89157
[15]	validation_0-mae:5923.27660
[16]	validation_0-mae:5689.96131
[17]	validation_0-mae:5482.51189
[18]	validation_0-mae:5310.63328
[19]	validation_0-mae:5142.00631
[20]	validation_0-mae

C:\Users\verga\AppData\Local\Temp\ipykernel_49172\3878269409.py:36: UserWarning: Could not infer format, so each element will be parsed individually, falling back to `dateutil`. To ensure parsing is consistent and as-expected, please specify a format.
  dt = pd.to_datetime(Xo["date"], errors="coerce")
C:\Users\verga\AppData\Local\Temp\ipykernel_49172\3878269409.py:36: UserWarning: Could not infer format, so each element will be parsed individually, falling back to `dateutil`. To ensure parsing is consistent and as-expected, please specify a format.
  dt = pd.to_datetime(Xo["date"], errors="coerce")


[36]	validation_0-mae:3446.36357
[37]	validation_0-mae:3397.56184
[38]	validation_0-mae:3349.82494
[39]	validation_0-mae:3302.31796
[40]	validation_0-mae:3258.87054
[41]	validation_0-mae:3224.09230
[42]	validation_0-mae:3184.99546
[43]	validation_0-mae:3142.67572
[44]	validation_0-mae:3110.69672
[45]	validation_0-mae:3063.44612
[46]	validation_0-mae:3024.03781
[47]	validation_0-mae:2999.90916
[48]	validation_0-mae:2972.70300
[49]	validation_0-mae:2944.21180
[50]	validation_0-mae:2927.64919
[51]	validation_0-mae:2903.16500
[52]	validation_0-mae:2886.25916
[53]	validation_0-mae:2859.17688
[54]	validation_0-mae:2834.66869
[55]	validation_0-mae:2817.28779
[56]	validation_0-mae:2792.53434
[57]	validation_0-mae:2767.83575
[58]	validation_0-mae:2757.91319
[59]	validation_0-mae:2747.30721
[60]	validation_0-mae:2733.70774
[61]	validation_0-mae:2710.38544
[62]	validation_0-mae:2682.16284
[63]	validation_0-mae:2671.29319
[64]	validation_0-mae:2664.80995
[65]	validation_0-mae:2654.64302
[66]	valid

### 4.1 ¿Qué es pruning y cómo impacta en el entrenamiento?

El pruning (poda de trials) es una técnica de optimización que detiene tempranamente aquellas configuraciones de hiperparámetros que, según el desempeño parcial observado durante el entrenamiento, no tienen proyección de mejorar al mejor resultado actual.
Impacto esperado: reduce tiempo de cómputo y recursos al evitar terminar trials poco prometedores, permitiendo explorar más combinaciones útiles dentro del mismo presupuesto de tiempo (misma ventana de 5 minutos), sin cambiar la métrica objetivo ni el pipeline final.

### 4.4 ¿Cómo cambian sus resultados con respecto a la sección anterior? ¿A qué se puede deber esto?
Al incorporar la técnica de pruning en la optimización de hiperparámetros, se observó una ligera variación en los resultados en comparación con la búsqueda bayesiana tradicional. El nuevo modelo alcanzó un MAE de 2028.31, muy cercano al 2012.35 obtenido previamente con Optuna sin pruning. Si bien el error aumentó marginalmente, el número total de trials realizados fue menor (267 frente a 321), lo que evidencia que el pruning logró reducir el tiempo de cómputo descartando tempranamente combinaciones poco prometedoras.

Este pequeño aumento en el MAE puede atribuirse a que algunos trials con potencial de mejora fueron podados antes de completar su entrenamiento, o a la naturaleza aleatoria del muestreo bayesiano. Aun así, el modelo con pruning mantiene un desempeño prácticamente equivalente, mostrando que la técnica es efectiva para acelerar la búsqueda de hiperparámetros sin comprometer significativamente la calidad predictiva.

## 5. Visualizaciones (5 puntos)

<p align="center">
  <img src="https://media.tenor.com/F-LgB1xTebEAAAAd/look-at-this-graph-nickelback.gif">
</p>


Satisfecho con su trabajo, Fiu le pregunta si es posible generar visualizaciones que permitan entender el entrenamiento de su modelo.

A partir del siguiente <a href = https://optuna.readthedocs.io/en/stable/tutorial/10_key_features/005_visualization.html#visualization>enlace</a>, genere las siguientes visualizaciones:

1. Gráfico de historial de optimización [1 punto]
2. Gráfico de coordenadas paralelas [1 punto]
3. Gráfico de importancia de hiperparámetros [1 punto]

Comente sus resultados:

4. ¿Desde qué *trial* se empiezan a observar mejoras notables en sus resultados? [0.5 puntos]
5. ¿Qué tendencias puede observar a partir del gráfico de coordenadas paralelas? [1 punto]
6. ¿Cuáles son los hiperparámetros con mayor importancia para la optimización de su modelo? [0.5 puntos]

In [ ]:
# 5. Visualizaciones con Optuna
from optuna.visualization import plot_optimization_history, plot_parallel_coordinate, plot_param_importances
study_to_plot = study_pruned  # (de la parte 4)

# 5.1️ Gráfico de historial de optimización
fig1 = plot_optimization_history(study_to_plot)
fig1.show()

# 5.2️ Gráfico de coordenadas paralelas
fig2 = plot_parallel_coordinate(study_to_plot)
fig2.show()

# 5.3️ Gráfico de importancia de hiperparámetros
fig3 = plot_param_importances(study_to_plot)
fig3.show()


c:\Users\verga\AppData\Local\Programs\Python\Python310\lib\site-packages\kaleido\_sync_server.py:11: UserWarning:




This means that static image generation (e.g. `fig.write_image()`) will not work.

Please upgrade Plotly to version 6.1.1 or greater, or downgrade Kaleido to version 0.2.1.




### 5.4 ¿Desde qué trial se empiezan a observar mejoras notables en los resultados?

A partir del gráfico de historial de optimización, se observa que las mejoras más marcadas en el valor objetivo (MAE) comienzan alrededor del trial 5 al 10, donde el error baja bruscamente desde valores superiores a 7000 hasta cerca de 2000.
Después de ese punto, las mejoras son graduales y se estabilizan en torno al MAE mínimo (~2000) a partir del trial 40, indicando que el optimizador ya había explorado una zona próxima al óptimo.

### 5.5 ¿Qué tendencias puede observar a partir del gráfico de coordenadas paralelas?

El gráfico de coordenadas paralelas muestra que los trials con menor error tienden a cumplir las siguientes condiciones:

learning_rate se mantiene en valores medios (~0.04 a 0.08), evitando tanto tasas demasiado bajas como altas.

max_depth y max_leaves alcanzan valores altos (cercanos a 8–10 y 80–100 respectivamente), lo que sugiere que una mayor complejidad del árbol ayuda al modelo a capturar patrones relevantes.

n_estimators suele ser elevado (600–900), reforzando que más iteraciones permiten una convergencia más fina.

reg_alpha y reg_lambda tienden a valores bajos, indicando que una regularización moderada o débil favorece el ajuste.

En conjunto, estas tendencias sugieren que el modelo rinde mejor cuando combina una tasa de aprendizaje intermedia con alta capacidad de modelado y regularización leve.

### 5.6 ¿Cuáles son los hiperparámetros con mayor importancia para la optimización de su modelo?

De acuerdo con el gráfico de importancia de hiperparámetros, los tres parámetros más influyentes en el rendimiento del modelo son:

ohe_min_frequency — representa el umbral mínimo de frecuencia de categorías en el OneHotEncoder. Su alta importancia (≈ 0.64) muestra que el tratamiento de variables categóricas tiene un fuerte impacto en la generalización.

learning_rate — controla la velocidad de aprendizaje del modelo (≈ 0.20), afectando directamente la convergencia y estabilidad del entrenamiento.

max_leaves — influye en la complejidad de los árboles generados (≈ 0.08), determinando la capacidad del modelo para capturar relaciones no lineales.

El resto de los hiperparámetros —como n_estimators, max_depth, reg_alpha, y reg_lambda— presentan una influencia menor, lo que sugiere que su ajuste fino tiene un impacto limitado frente a las variables mencionadas.

## 6. Síntesis de resultados (3 puntos)

Finalmente:

1. Genere una tabla resumen del MAE en el conjunto de validación obtenido en los 5 modelos entrenados desde Baseline hasta XGBoost con Constraints, Optuna y Prunning. [1 punto]
2. Compare los resultados de la tabla y responda, ¿qué modelo obtiene el mejor rendimiento? [0.5 puntos]
3. Cargue el mejor modelo, prediga sobre el conjunto de **test** y reporte su MAE. [0.5 puntos]
4. ¿Existen diferencias con respecto a las métricas obtenidas en el conjunto de validación? ¿Porqué puede ocurrir esto? [1 punto]

In [ ]:
# 6. Síntesis de resultados
# 6.1 Tabla resumen del MAE en validación
resultados_mae = pd.DataFrame({
    "Modelo": [
        "Baseline DummyRegressor",
        "XGBRegressor (Default)",
        "XGBRegressor (Monotonicidad)",
        "XGBRegressor (Optuna)",
        "XGBRegressor (Optuna + Prunning)"
    ],
    "MAE Validación": [
        13298.497767,   # obtenido anteriormente
        2413.171360,    # baseline XGB default
        2499.159773,    # XGB con monotonicidad
        2012.350371,    # mejor con Optuna
        2028.307507     # con Optuna + pruning
    ]
})

display(resultados_mae)

# 6.2 Identificacion del mejor modelo
mejor_modelo = resultados_mae.loc[resultados_mae["MAE Validación"].idxmin(), "Modelo"]
print(f"\nEl mejor rendimiento en validación lo obtiene: {mejor_modelo}")

# 6.3 Cargar el mejor modelo (Optuna) y evaluar en conjunto de test
modelo_final = joblib.load("xgb_optuna.pkl")
y_pred_test = modelo_final.predict(X_test)
mae_test = mean_absolute_error(y_test, y_pred_test)

print(f"\nMAE en conjunto de test (modelo Optuna): {mae_test:.4f}")


,Modelo,MAE Validación
0,Baseline DummyRegressor,13298.497767
1,XGBRegressor (Default),2413.171360
2,XGBRegressor (Monotonicidad),2499.159773
3,XGBRegressor (Optuna),2012.350371
4,XGBRegressor (Optuna + Prunning),2028.307507



El mejor rendimiento en validación lo obtiene: XGBRegressor (Optuna)

MAE en conjunto de test (modelo Optuna): 2089.2285


C:\Users\verga\AppData\Local\Temp\ipykernel_49172\3878269409.py:36: UserWarning:

Could not infer format, so each element will be parsed individually, falling back to `dateutil`. To ensure parsing is consistent and as-expected, please specify a format.



### 6.4 Análisis final

El modelo con mejor rendimiento en el conjunto de validación fue el XGBRegressor optimizado con Optuna, con un MAE de 2012.35.
En comparación con las demás configuraciones, este modelo logró reducir significativamente el error frente al XGB baseline (2413.17) y al modelo con restricción monotónica (2499.16).

Al evaluar el mejor modelo sobre el conjunto de test, se obtuvo un MAE de 2089.23, ligeramente superior al obtenido en validación.
Esta diferencia es esperable y normal, ya que el conjunto de test representa datos completamente nuevos que el modelo no ha visto durante el entrenamiento ni la optimización.

Estas pequeñas variaciones pueden deberse a:

Diferencias leves en la distribución de las variables entre validación y test.

Aleatoriedad en el muestreo y en la inicialización del modelo.

Un posible ajuste fino hacia el conjunto de validación debido al proceso de búsqueda de hiperparámetros.

En general, el resultado confirma que el modelo optimizado con Optuna mantiene una buena capacidad de generalización, siendo el más eficiente entre todas las versiones entrenadas.

# Conclusión
Exito!
<p align="center">
  <img src="https://i.pinimg.com/originals/55/3d/42/553d42bea9b10e0662a05aa8726fc7f4.gif">
</p>